# EMG time-frequency power around button presses

This notebook follows EMG_timefrequency_plot_subject.ipynb. For each participant it computes a Morlet time-frequency power map for every EMG channel in the −0.5 to +0.5 s window around button presses. Each participant's map is baseline corrected using the average power from −1 to 0 s around each trial's cue-time reference.

The participant maps are averaged to make one group time-frequency plot for each EMG channel. The group average is calculated from each participant's baseline-corrected log-power map, so each included participant has equal weight.

import packages for file reading, MEG data handling and plotting.

In [ ]:
import h5py
import mne
import numpy as np
import matplotlib.pyplot as plt

define parameters and the subject list. IDs in excluded_subjects are skipped.

In [ ]:
sub_ID = ['05','06','08','09','10','11','13','14','15','16','17','18','20','21','24','25','28']  # subject IDs
excluded_subjects = ['11','16','21']  # participants with questionable data

ch_names = ['EEG057','EEG058','EEG059','EEG060']  # EMG channels
freqs = np.arange(1, 51, 1)                       # frequency bins in Hz
n_cycles = freqs / 10                             # Morlet cycles per frequency
press_time_wind = np.array([-0.5, 0.5])           # time window around each button press
baseline_time_wind = np.array([-1, 0])            # baseline window before each trial
delay = 2.8                                        # cue-time offset within each trial
trial_time = 14.8                                  # trial duration in seconds
n_press_slots = 5                                  # button press slots per trial

load each participant's files

filename_loader now follows the same file paths as MEG_powerplot_all: behavioral files are under /beh_data/sXX, and the FIF file is under /sXX. For IDs with a leading zero, the folder stays zero-padded (s05) while the filename uses the unpadded number (Subject5). The behavioral arrays are transposed into trials-by-press-slots layout before the cue duration is added to each button time.

Subject 16 has 240 behavioral rows but 222 saved epochs. When those counts occur together, the first 222 rows are used and the final 18 rows are ignored. Other trial-count mismatches stop with an error.

In [ ]:
def filename_loader(ID):
    # define file directories for behavioural data and the EMG epoch file
    subject_folder = "//rdp.arc.ucl.ac.uk/ritd-ag-project-rd01pz-dbush99/Katja Button Press Data"
    button_press_folder = '/beh_data/s' + str(ID)
    # use the same FIF filename pattern as MEG_powerplot_all
    if ID.startswith('0'):
        spliced_id = ID[1:]
        EMG_file = '/s0' + spliced_id + '/Subject' + spliced_id + '_spm_mne-epo'
    else:
        EMG_file = '/s' + str(ID) + '/Subject' + str(ID) + '_spm_mne-epo'

    BP1 = subject_folder + button_press_folder + "/behDataMEG.mat"
    BP2 = subject_folder + button_press_folder + "/targetMEG.mat"
    EMG = subject_folder + EMG_file + '.fif'

    return BP1, BP2, EMG

calculate one participant's time-frequency power

The steps in this section match the subject notebook: combine trials in time, make press and baseline epochs, compute Morlet power, then divide by the baseline and take the natural logarithm.

In [ ]:
def participant_time_frequency(ID):
    BP_filepath_1, BP_filepath_2, EMG_filepath = filename_loader(ID)

    # load button press times and press identity codes
    with h5py.File(BP_filepath_1, 'r') as f:
        timings = f['A/timing'][:]
        presses = f['A/press'][:]

    # load cue duration for each trial
    with h5py.File(BP_filepath_2, 'r') as f:
        cuedur = f['T/cueDur'][:]

    # transpose the arrays to trials-by-press-slots layout
    timings = timings.T
    presses = presses.T
    cuedur = np.asarray(cuedur).reshape(-1)

    # add each trial's cue duration to all of its press slots
    if timings.shape[0] != cuedur.size:
        raise ValueError('Button times and cue durations have different trial counts')
    if timings.shape[1] != n_press_slots:
        raise ValueError('Unexpected number of button press slots')
    press_times = (timings + cuedur[:, None]) / 1000.0

    # load the epoched EMG recording
    EMG_data = mne.read_epochs(EMG_filepath, preload=True)

    if ID == '16' and len(EMG_data) == 222 and press_times.shape[0] == 240:
        print('Subject 16: using the first 222 behavioral trial rows.')
        press_times = press_times[:len(EMG_data), :]
    elif len(EMG_data) != press_times.shape[0]:
        raise ValueError(
            'Subject ' + str(ID) + ' has ' + str(len(EMG_data))
            + ' EMG epochs but ' + str(press_times.shape[0])
            + ' behavioral trials'
        )

    # isolate the EMG channels and combine trials end-to-end in a RawArray
    EMG_channels = EMG_data.get_data(picks=ch_names)
    sfreq = EMG_data.info['sfreq']
    n_epochs, n_channels, n_times = EMG_channels.shape
    data = EMG_channels.transpose([1, 0, 2]).reshape(
        (n_channels, n_epochs * n_times)
    )

    ch_types = ['emg'] * len(ch_names)
    info = mne.create_info(ch_names=ch_names, ch_types=ch_types, sfreq=sfreq)
    raw_epoch = mne.io.RawArray(data, info)

    # shift press times onto the combined timeline; baseline events start at cue time
    trial_offsets = np.arange(n_epochs) * trial_time
    press_times_shifted = press_times + trial_offsets[:, None]
    trial_cutoff_times = trial_offsets + delay

    # flatten press times and remove missing values
    press_times_shifted = press_times_shifted.flatten()
    valid = np.isfinite(press_times_shifted)
    print('Removed', int((~valid).sum()), 'missing/non-finite press times')
    press_times_shifted = press_times_shifted[valid]

    # convert event times from seconds to sample indices
    samples_pt = np.round(press_times_shifted * sfreq).astype(int)
    samples_trial = np.round(trial_cutoff_times * sfreq).astype(int)

    # define the button-press and trial-cutoff event arrays
    bp_events = np.column_stack([
        samples_pt,
        np.zeros(len(samples_pt), dtype=int),
        np.zeros(len(samples_pt), dtype=int),
    ])
    trial_events = np.column_stack([
        samples_trial,
        np.zeros(len(samples_trial), dtype=int),
        np.zeros(len(samples_trial), dtype=int),
    ])

    # make epochs around button presses and trial baselines
    tfr_epochs = mne.Epochs(
        raw_epoch,
        bp_events,
        tmin=press_time_wind[0],
        tmax=press_time_wind[1],
        baseline=None,
        preload=True,
    )
    baseline_epochs = mne.Epochs(
        raw_epoch,
        trial_events,
        tmin=baseline_time_wind[0],
        tmax=baseline_time_wind[1],
        baseline=None,
        preload=True,
        verbose=False,
    )

    # calculate Morlet time-frequency power for each participant
    power = tfr_epochs.compute_tfr(
        method='morlet',
        freqs=freqs,
        n_cycles=n_cycles,
        picks=ch_names,
        average=True,
        return_itc=False,
    )
    baseline_tfr = baseline_epochs.compute_tfr(
        method='morlet',
        freqs=freqs,
        n_cycles=n_cycles,
        picks=ch_names,
        average=True,
        return_itc=False,
        verbose=False,
    )

    # subtract the baseline mean for each channel and frequency in log space
    baseline_mean = baseline_tfr.data.mean(axis=-1, keepdims=True)
    power.data = np.log(power.data / baseline_mean)

    print('Subject', ID, '- press epochs:', len(tfr_epochs))
    return power.data, power.times, power.freqs

average the time-frequency power across participants

Each participant's channel maps are averaged only after the subject-level baseline correction. This keeps the participant weighting equal even if participants have different numbers of valid button presses.

The time and frequency grids and the EMG channel order are checked before averaging so that values line up across participants.

In [ ]:
subjects_to_use = [ID for ID in sub_ID if ID not in excluded_subjects]

if len(subjects_to_use) == 0:
    raise ValueError('No subjects remain after applying excluded_subjects.')

print('Subjects included:', subjects_to_use)

subject_power = []
subjects_used = []
group_times = None
group_freqs = None

for ID in subjects_to_use:
    power_data, power_times, power_freqs = participant_time_frequency(ID)

    if group_times is None:
        group_times = power_times
        group_freqs = power_freqs
    elif not np.allclose(power_times, group_times) or not np.allclose(power_freqs, group_freqs):
        raise ValueError('Time or frequency bins differ for subject ' + str(ID))

    subject_power.append(power_data)
    subjects_used.append(ID)

# average the participant maps for each EMG channel
subject_power = np.asarray(subject_power)
group_power = subject_power.mean(axis=0)

print('Participants averaged:', subjects_used)
print('Group time-frequency array shape:', group_power.shape)

plot the group time-frequency power for each EMG channel

The color scale is shared across all four channels. It is set from the 80th percentile of the absolute group-map values within the displayed time window, as in the subject notebook.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
time_window = np.array([-0.4, 0.1])

visible_times = (group_times >= time_window[0]) & (group_times <= time_window[1])
vlim = np.nanpercentile(np.abs(group_power[:, :, visible_times]), 80)

for ch_idx, ax in enumerate(axes.flat):
    image = ax.pcolormesh(
        group_times,
        group_freqs,
        group_power[ch_idx],
        shading='auto',
        cmap='jet',
    )
    ax.axvline(0, color='black', linestyle='--', linewidth=1)
    ax.set_title(ch_names[ch_idx])
    ax.set_xlabel('Time relative to button press (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlim(time_window[0], time_window[1])
    ax.set_ylim(0, 50)
    fig.colorbar(image, ax=ax, label='log(Power change from baseline)')
    image.set_clim(vmin=-vlim, vmax=vlim)

fig.suptitle('Mean EMG time-frequency power across participants')
fig.tight_layout()
plt.show()